# Phase 8 B v4 — episode, eligibility, rider, dan audit

Notebook ini mereplay pengukuran Phase 8 yang telah tersimpan. Tidak menjalankan detector, tracker, encoder ReID, atau enhancement. ReID tetap bagian arsitektur; konfigurasi vehicle-ReID final masih terbuka.

Pilot default adalah **seluruh 9.128 frame TGDI-WEEKDAY-SIANG**, agar state tidak dimulai di tengah klip. Kasus pilot sudah dipakai untuk development; hasilnya bukan blind hold-out atau ukuran akurasi.

B v3, input model, dan bukti Phase 9 tidak ditimpa. Branch `machine` memakai review lama yang sama dengan B v3. Branch `review_assisted` menambahkan adjudikasi AI yang dibatasi dan diikat ke bukti Phase 9; itu bukan kemampuan otomatis baru dan bukan label pengguna baru. Jangan menjumlahkan kedua branch.


In [ ]:
from pathlib import Path
import json, sys, unittest

# Checkout harus memuat hasil revisi ini; kredensial tidak dimasukkan ke notebook.
REPO_ROOT = Path(globals().get("REPO_ROOT", "/kaggle/working/traffic-vehicle-counter-IDM-LOC")).resolve()
SOURCE_RUN_ROOT = globals().get("SOURCE_RUN_ROOT", None)
SOURCE_VIDEO_ROOT = globals().get("SOURCE_VIDEO_ROOT", None)
INPUT_ROOT = globals().get("INPUT_ROOT", "/kaggle/input")
VIDEO_IDS = globals().get("VIDEO_IDS", ["weekday_siang"])
MODE = globals().get("MODE", "prepare")  # prepare: validasi input saja; run: replay counting baru.
BRANCHES_TO_RUN = globals().get("BRANCHES_TO_RUN", ["machine", "review_assisted"])
RENDER_ANNOTATED_VIDEO = globals().get("RENDER_ANNOTATED_VIDEO", False)
OUTPUT_ROOT = globals().get("OUTPUT_ROOT", "/kaggle/working/results/phase_8_b_v4")
POLICY_PATH = globals().get("POLICY_PATH", None)  # Alternatif profil eksplisit harus dipertahankan byte-nya.

sys.path.insert(0, str(REPO_ROOT / "notebooks"))


## Validasi software dan input

Lampirkan dataset `chrisbiran/tgdi-traffic-videos` dan output donor `chrisbiran/phase-8-tgdi-full-video-research` di Kaggle, atau isi dua root sumber yang eksplisit pada lingkungan lain. Auto-discovery memilih hanya manifest donor yang SHA-nya cocok. CPU cukup untuk replay ini; GPU tidak diperlukan.

Tes memeriksa invariant sumber, episode, role, counting dan renderer. Tes tidak menghasilkan bukti performa model. Notebook berhenti jika ada kegagalan.


In [ ]:
modules = ["phase_8_b_v3_counting_tests", "phase_8_b_v3_annotation_tests", "phase_8_b_v3_batch_tests",
    "phase_8_b_v4_identity_tests", "phase_8_b_v4_counting_tests", "phase_8_b_v4_annotation_tests",
    "phase_8_b_v4_batch_tests"]
suite = unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromName(name) for name in modules)
validation = unittest.TextTestRunner(verbosity=1).run(suite)
assert validation.wasSuccessful() and not validation.skipped, "Validation failed or skipped"


In [ ]:
from phase_8_b_v4_batch import prepare_b_v4_batch, run_b_v4_batch
assert MODE in ("prepare", "run")
prepared_inputs = []
for branch in BRANCHES_TO_RUN:
    prepared = prepare_b_v4_batch(repo_root=REPO_ROOT, source_run_root=SOURCE_RUN_ROOT,
        source_video_root=SOURCE_VIDEO_ROOT, input_root=INPUT_ROOT, video_ids=VIDEO_IDS,
        branch=branch, policy_path=POLICY_PATH)
    prepared_inputs.append({"branch": branch, "selected_video_ids": prepared["selected_video_ids"],
        "selected_video_count": len(prepared["jobs"]), "policy_sha256": prepared["policy_sha256"],
        "effective_policy_sha256": prepared["effective_policy_sha256"],
        "review_overlay_sha256": prepared["review_overlay_sha256"]})
print(json.dumps(prepared_inputs, indent=2))


## Replay terpisah per branch

Untuk menjalankan, ubah `MODE` menjadi `"run"`. Untuk seluruh delapan video, set `VIDEO_IDS` secara eksplisit ke `weekday_pagi`, `weekday_siang`, `weekday_sore`, `weekday_malam`, `weekend_pagi`, `weekend_siang`, `weekend_sore`, `weekend_malam`. Jangan mengklaim delapan video selesai dari pilot satu video.

Margin 25px, timer 60 **frame sumber** sejak observasi terakhir di polygon asal, geometri native640×480, dan aturan kategori tetap. Numeric gates asosiasi adalah kontrol eksperimen yang eksplisit, bukan default produksi terverifikasi. Role/ID review tidak mengubah BC sumber dan tidak mengesahkan crossing selama gap.


In [ ]:
reports = []
if MODE == "run":
    for branch in BRANCHES_TO_RUN:
        report = run_b_v4_batch(repo_root=REPO_ROOT, source_run_root=SOURCE_RUN_ROOT,
            source_video_root=SOURCE_VIDEO_ROOT, input_root=INPUT_ROOT, video_ids=VIDEO_IDS,
            output_root=OUTPUT_ROOT, render=RENDER_ANNOTATED_VIDEO, branch=branch,
            policy_path=POLICY_PATH)
        reports.append(report)
    print(json.dumps([{key: report.get(key) for key in ("branch", "status", "output_directory",
        "frames_processed", "operational_total", "pending_unique_unit_episode_passages",
        "new_detector_tracker_reid_execution")} for report in reports], indent=2))
else:
    print("PREPARE selesai. Replay counting dan renderer belum dijalankan.")


## Keluaran dan review

Setiap run baru menyimpan config efektif, review lama, overlay (jika ada), snapshot executable/notebook dan retention manifest. Setiap video berisi `counting_result.json`, `episode_audit.json`, `role_audit.json`, `handoff_audit.json`, `eligibility_audit.json`, `adapter_audit.json`, `frame_assignments.json`, `pending_groups.json`, `person_role_review_queue.json`, dan `comparison_b_v3.json`.

Jika render aktif, simpan MP4 serta laporan renderer/layout untuk memeriksa raw ID, parent rider, status passage, sumber crossing dan panel. Event ID berbeda antarversi: bandingkan kendaraan/passage fisik beserta raw ID, interval dan measurement refs; kesamaan raw ID saja bukan bukti identitas.

Periksa khusus: satu raw ID pindah objek (166/185), fragmen muncul di sisi tujuan menjauh (856, prehistory tetap unresolved), interval rider857, pedestrian54 pada branch review-assisted, pedestrian3025 di luar ROI, truck4149 saat uji regresi seluruh delapan video. Periksa juga kendaraan berdekatan, kendaraan diam berganti ID, genuine return dan hitungan yang terlewat.

Penurunan pending, event atau timeout saja tidak membuktikan peningkatan. Accepted terukur dan estimasi tetap dipisah. Pending dapat selesai sebagai tidak eligible/+0. Validasi malam lintas kondisi, cakupan pedestrian dekat kamera, konfigurasi ReID final dan parameter produksi tetap terbuka. Phase10 belum dimulai.
